In [ ]:
!nvidia-smi

Mon Mar  9 19:37:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# GSM8K 0-Shot Evaluation — LLaMA-3 8B (vLLM)

In [ ]:
!pip install "protobuf==5.29.5"
!pip install -q vllm datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 8.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.

In [ ]:
!pip install "protobuf==5.29.5"

  Using cached protobuf-5.29.5-cp38-abi3-manylinux2014_x86_64.whl.metadata (592 bytes)
Using cached protobuf-5.29.5-cp38-abi3-manylinux2014_x86_64.whl (319 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.5
    Uninstalling protobuf-6.33.5:
      Successfully uninstalled protobuf-6.33.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.17.0 requires protobuf!=6.30.*,!=6.31.*,!=6.32.*,!=6.33.0.*,!=6.33.1.*,!=6.33.2.*,!=6.33.3.*,!=6.33.4.*,>=5.29.6, but you have protobuf 5.29.5 which is incompatible.
grpcio-reflection 1.78.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 5.29.5 which is incompatible.
google-adk 1.26.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.26.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
import re
import json
import time
import numpy as np
from datasets import load_dataset

print("Loading GSM8K test set...")
ds = load_dataset("gsm8k", "main", split="test")

print(f"Dataset size: {len(ds)} examples")
print(f"Columns: {ds.column_names}")


def extract_answer_number(text):
    """Extract the last number from text. Handles GSM8K '####' marker and commas."""
    if "####" in text:
        text = text.split("####")[-1]
    text = text.replace(",", "")
    matches = re.findall(r"-?\d+\.?\d*", text)
    return matches[-1] if matches else None

Loading GSM8K test set...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset size: 1319 examples
Columns: ['question', 'answer']


In [ ]:
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
MAX_NEW_TOKENS = 512
EVAL_SIZE = len(ds)

print({
    "model": MODEL_NAME,
    "eval_size": EVAL_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
})

{'model': 'meta-llama/Meta-Llama-3-8B-Instruct', 'eval_size': 1319, 'max_new_tokens': 512}


# Model

In [ ]:
if __name__ == '__main__':
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    print(f"Loading {MODEL_NAME} with vLLM...")
    llm = LLM(
        model=MODEL_NAME,
        dtype="float16",
        gpu_memory_utilization=0.95,
        max_model_len=4096,
        enforce_eager=False,
    )

    sampling_params = SamplingParams(
        temperature=0,
        top_k=20,
        max_tokens=MAX_NEW_TOKENS,
    )

    print("Model loaded successfully!")

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Loading meta-llama/Meta-Llama-3-8B-Instruct with vLLM...
INFO 03-09 19:40:47 [utils.py:238] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'model': 'meta-llama/Meta-Llama-3-8B-Instruct'}


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

INFO 03-09 19:41:10 [model.py:531] Resolved architecture: LlamaForCausalLM
WARNING 03-09 19:41:10 [model.py:1892] Casting torch.bfloat16 to torch.float16.
INFO 03-09 19:41:10 [model.py:1554] Using max model len 4096
INFO 03-09 19:41:10 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-09 19:41:10 [vllm.py:747] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

WARNING 03-09 19:41:11 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 03-09 19:43:11 [llm.py:388] Supported tasks: ['generate']
Model loaded successfully!


# Output Directory

In [8]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/Colab_Data/GSM8K/Llama3_8B_GSM8K_Eval_vLLM"
except Exception:
    DRIVE_SAVE_DIR = os.path.abspath("./llama3_8b_gsm8k_outputs")

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Saving results to: {DRIVE_SAVE_DIR}")

Mounted at /content/drive
Saving results to: /content/drive/MyDrive/Colab_Data/GSM8K/Llama3_8B_GSM8K_Eval_vLLM


# Checkpoint Setup & Prompt Construction

In [9]:
CHECKPOINT_FILE = os.path.join(DRIVE_SAVE_DIR, "llama3_8b_gsm8k_vllm_checkpoint.jsonl")
OUTPUTS_CACHE   = os.path.join(DRIVE_SAVE_DIR, "llama3_8b_gsm8k_vllm_raw_outputs.json")

results = []
if os.path.exists(CHECKPOINT_FILE):
    print(f"Found checkpoint: {CHECKPOINT_FILE}")
    with open(CHECKPOINT_FILE) as f:
        for line in f:
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    print(f"Loaded {len(results)} previously evaluated examples.")
else:
    print("No checkpoint found — starting from scratch.")

subset = ds.select(range(EVAL_SIZE))
questions = subset["question"]
ground_truths = [extract_answer_number(a) for a in subset["answer"]]

already_done = len(results)
remaining_qs = questions[already_done:]
print(f"{already_done} already done, {len(remaining_qs)} remaining.")

No checkpoint found — starting from scratch.
0 already done, 1319 remaining.


In [10]:
prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": q}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for q in remaining_qs
]

print(f"Built {len(prompts)} prompts.")
if prompts:
    print(f"\nExample prompt (first 400 chars):\n{prompts[0][:400]}")
else:
    print("All requested examples are already processed.")

Built 1319 prompts.

Example prompt (first 400 chars):
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




# Generation

In [11]:
if prompts:
    print(f"Generating responses for {len(prompts)} prompts...")

    gen_start = time.time()
    outputs = llm.generate(prompts, sampling_params)
    gen_time = time.time() - gen_start

    total_new_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput = total_new_tokens / gen_time if gen_time > 0 else None

    print("\nGeneration complete.")
    print(f"  Time:       {gen_time / 60:.1f} min")
    print(f"  Tokens:     {total_new_tokens:,}")
    if throughput:
        print(f"  Throughput: {throughput:.1f} tokens/sec")

    raw_cache = [
        {
            "question_idx": already_done + i,
            "question": remaining_qs[i],
            "ground_truth": ground_truths[already_done + i],
            "response": o.outputs[0].text,
            "n_tokens": len(o.outputs[0].token_ids),
        }
        for i, o in enumerate(outputs)
    ]

    with open(OUTPUTS_CACHE, "w") as f:
        json.dump(raw_cache, f)
    print(f"Raw outputs cached to: {OUTPUTS_CACHE}")
else:
    print("No prompts to generate — loading cached outputs.")
    if os.path.exists(OUTPUTS_CACHE):
        with open(OUTPUTS_CACHE) as f:
            raw_cache = json.load(f)
    else:
        raw_cache = []
    gen_time = None
    throughput = None
    total_new_tokens = sum(r.get("n_tokens", 0) for r in raw_cache)

print(f"Raw items available for scoring: {len(raw_cache)}")

Generating responses for 1319 prompts...


Rendering prompts:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


Generation complete.
  Time:       1.8 min
  Tokens:     113,419
  Throughput: 1056.2 tokens/sec
Raw outputs cached to: /content/drive/MyDrive/Colab_Data/GSM8K/Llama3_8B_GSM8K_Eval_vLLM/llama3_8b_gsm8k_vllm_raw_outputs.json
Raw items available for scoring: 1319


# Scoring & Final Metrics

In [12]:
new_results = []
for item in raw_cache:
    idx = item["question_idx"]
    response = item["response"]

    pred = extract_answer_number(response)
    gt = ground_truths[idx]

    is_correct = False
    if pred is not None and gt is not None:
        try:
            is_correct = float(pred) == float(gt)
        except ValueError:
            pass

    res = {
        "question_idx": idx,
        "prediction": pred,
        "ground_truth": gt,
        "raw_response": response.strip(),
        "is_correct": is_correct,
        "is_unknown": pred is None,
        "new_tokens": item["n_tokens"],
    }
    new_results.append(res)

if new_results:
    with open(CHECKPOINT_FILE, "a") as f:
        for r in new_results:
            f.write(json.dumps(r) + "\n")
    results.extend(new_results)

print(f"Total scored results: {len(results)}")

correct_count = sum(1 for r in results if r["is_correct"])
unknown_count = sum(1 for r in results if r["is_unknown"])
all_new_tokens = sum(r["new_tokens"] for r in results)
accuracy = correct_count / len(results) if results else 0

known_preds = [r for r in results if not r["is_unknown"]]
accuracy_known = (
    sum(1 for r in known_preds if r["is_correct"]) / len(known_preds)
    if known_preds
    else 0
)

final_metrics = {
    "method": "0_shot_vllm",
    "model": MODEL_NAME,
    "dataset": "gsm8k/main:test",
    "eval_size": len(results),
    "accuracy": accuracy,
    "accuracy_known_only": accuracy_known if known_preds else "N/A",
    "unknown_frac": unknown_count / len(results) if results else 0,
    "total_new_tokens": all_new_tokens,
    "gen_tokens_per_sec": throughput if throughput is not None else "N/A (loaded from cache)",
    "total_gen_time_min": gen_time / 60 if gen_time is not None else "N/A (loaded from cache)",
}

print("\n=== Final Metrics ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

METRICS_FILE = os.path.join(DRIVE_SAVE_DIR, "final_metrics.json")
with open(METRICS_FILE, "w") as f:
    json.dump(final_metrics, f, indent=2)
print(f"\nMetrics saved to: {METRICS_FILE}")

Total scored results: 1319

=== Final Metrics ===
  method: 0_shot_vllm
  model: meta-llama/Meta-Llama-3-8B-Instruct
  dataset: gsm8k/main:test
  eval_size: 1319
  accuracy: 0.7725549658832449
  accuracy_known_only: 0.7725549658832449
  unknown_frac: 0.0
  total_new_tokens: 113419
  gen_tokens_per_sec: 1056.1903971860793
  total_gen_time_min: 1.7897499084472657

Metrics saved to: /content/drive/MyDrive/Colab_Data/GSM8K/Llama3_8B_GSM8K_Eval_vLLM/final_metrics.json
